In [11]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *
from test_utils import *

In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
seed_number = 42
import random

random.seed(seed_number)
np.random.seed(seed_number)

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import dice_ml
from dice_ml.utils import helpers

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

names = ['existingchecking', 'duration', 'credithistory', 'purpose', 'creditamount', 
         'savings', 'employmentsince', 'installmentrate', 'statussex', 'otherdebtors', 
         'residencesince', 'property', 'age', 'otherinstallmentplans', 'housing', 
         'existingcredits', 'job', 'peopleliable', 'telephone', 'foreignworker', 'target']
dataset = pd.read_csv(f"{ugce_dir}/data/GermanCredit.data", sep=' ', header=None ,names = names)
TARGET_COLUMN = 'target'
dataset[TARGET_COLUMN] = LabelEncoder().fit_transform(dataset[TARGET_COLUMN])
datasetX = dataset.drop(columns=[TARGET_COLUMN])
target = dataset[TARGET_COLUMN]

x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = datasetX.select_dtypes(include=[np.number]).columns.tolist()
categorical = x_train.columns.difference(numerical)

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical),
        ('cat', categorical_transformer, categorical)])

model = RandomForestClassifier(random_state=42)

model = Pipeline(steps=[('preprocessor', transformations),
                      ('classifier', model)])

model.fit(x_train, y_train)

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

negative_instances = x_test[model.predict(x_test) == 0]
instances_to_explain = negative_instances
print("Number of instances to explain: ", len(instances_to_explain))

Accuracy:  0.75
Number of instances to explain:  166


In [15]:
len(numerical), len(categorical)

(7, 13)

In [16]:
datasetName = 'GermanCredit'

In [17]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

In [ ]:
import warnings
import time
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# Import DiCE
import dice_ml
d = dice_ml.Data(dataframe=dataset, continuous_features=numerical, outcome_name=TARGET_COLUMN)
backend = 'sklearn'
m = dice_ml.Model(model=model, backend=backend)

features_to_vary = [col for col in datasetX.columns if col not in ["age", "statussex", "foreignworker"]]

explainers = []
for i in range(5):
    exp_genetic = dice_ml.Dice(d, m, method='genetic')
    dice_exp_genetic = exp_genetic.generate_counterfactuals(
        instances_to_explain, total_CFs=1, desired_class="opposite",
        features_to_vary=features_to_vary)
    explainers.append(dice_exp_genetic)

import os
import pickle
results_dir = f'{UGCE_dir}/results/dice_objects/{datasetName}'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(explainers, open(f'{results_dir}/dice_exp_genetic.pkl', 'wb'))

  0%|          | 0/166 [00:00<?, ?it/s]

100%|██████████| 166/166 [00:58<00:00,  2.82it/s]


In [9]:
# load explainers
import pickle
results_dir = f'{UGCE_dir}/results/dice_objects/{datasetName}'
explainers = pickle.load(open(f'{results_dir}/dice_exp_genetic.pkl', 'rb'))

In [ ]:
aggregate_results_DICE_baseline(iea, instances_to_explain, explainers, TARGET_COLUMN)

Full Time: mean = 18.6117, std = 15.4083
Generations: mean = 7.5476, std = 1.2898
Coverage: mean = 52.4096, std = 39.6347
Proximity Loss: mean = 0.0849, std = 0.0062
Sparsity: mean = 0.0192, std = 0.0007


# UGCE

## From Scratch

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

constraints = {
    "age":'-',
    "statussex":'-',
    "foreignworker":'-'
}

import time
start_time = time.time()
strategy = "fix_population_update_fitness"

results_baseline_explainer = []
for i in range(5):
    results_baseline = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=False, constraints=constraints,
        initial_population_variability=0.7, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=1,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=20, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints={}, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric='weighted_l1',
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_baseline_explainer.append(results_baseline)

import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/GermanCredit/baseline'
os.makedirs(results_dir, exist_ok=True)
strategy = "fix_population_update_fitness"
pickle.dump(results_baseline_explainer, open(f'{results_dir}/results_baseline{strategy}.pkl', 'wb'))

100%|██████████| 166/166 [00:22<00:00,  7.27it/s]


In [ ]:
from test_utils import *
aggregate_results_baseline(iea, results_baseline_explainer)

Full Time: mean = 22.4405, std = 0.1491
Generations: mean = 6.2205, std = 0.0677
Coverage: mean = 98.8095, std = 0.0000
Proximity Loss: mean = 0.0855, std = 0.0046
Sparsity: mean = 0.0209, std = 0.0002


In [8]:
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/baseline'
strategy = "fix_population_update_fitness"
with open(f'{results_dir}/results_baseline{strategy}.pkl', "rb") as file:
    results_baseline_explainer = pickle.load(file)

## Dynamic

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    "age":'i',
    "statussex":'i',
    "foreignworker":'i'
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints

results_incremental_explainer = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=1,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/GermanCredit/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer, open(f'{results_dir}/results_incremental{strategy}.pkl', 'wb'))

100%|██████████| 166/166 [00:44<00:00,  3.73it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [00:45<00:00,  3.68it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [00:41<00:00,  4.00it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [00:44<00:00,  3.75it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [00:45<00:00,  3.67it/s]

Empty intermediate counter: 0


In [158]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer, verbose=True)

Full Time: mean = 18.27, std = 0.68
Generations: mean = 6.03, std = 0.03
Coverage: mean = 95.36, std = 0.69
Proximity Loss: mean = 0.11, std = 0.01
Sparsity: mean = 0.02, std = 0.00
Intermediate Best Distances: mean = 0.07, std = 0.00


(18.270184564590455,
 6.032643584401289,
 95.35714285714286,
 0.11093073037870697,
 0.02197991611958487,
 0.06819445963334894)

In [154]:
## load results
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
strategy = "fix_population_update_fitness"
with open(f'{results_dir}/results_incremental{strategy}.pkl', "rb") as file:
    results_incremental_explainer = pickle.load(file)

# Assess the statistical Importance of correcting the population rather than starting with a random population

## Fixed population

In [160]:
total_time_dynamic_arr_for_ttest=[]
total_generations_arr_for_ttest=[]
total_cfes_found_arr_for_ttest=[]
total_proximity_loss_arr_for_ttest=[]
total_sparsity_arr_for_ttest=[]
total_best_intermediate_best_dist_arr_for_ttest=[]

full_times = []
coverages = []
distances = []
l1s = []
proximities = []
sparsities = []
generation_counts = []
intermediate_best_distances = []

for explainer in results_incremental_explainer:
    forttest, foravgs = stats_incremental(iea, explainer, return_matrices=True)
    time_dynamic_arr, generations_arr, cfes_found_arr, proximity_loss_arr, sparsity_arr, best_intermediate_best_dist_arr = forttest
    time_dynamic, avg_generations, avg_cfes_found, avg_l2, avg_l1, avg_proximity_loss, avg_sparsity, avg_best_intermediate_best_dist = foravgs
    
    total_time_dynamic_arr_for_ttest.extend(time_dynamic_arr)
    total_generations_arr_for_ttest.extend(generations_arr)
    total_cfes_found_arr_for_ttest.extend(cfes_found_arr)
    total_proximity_loss_arr_for_ttest.extend(proximity_loss_arr)
    total_sparsity_arr_for_ttest.extend(sparsity_arr)
    total_best_intermediate_best_dist_arr_for_ttest.extend(best_intermediate_best_dist_arr)

    full_times.append(time_dynamic)
    coverages.append(avg_cfes_found)
    distances.append(avg_l2)
    l1s.append(avg_l1)
    proximities.append(avg_proximity_loss)
    sparsities.append(avg_sparsity)
    generation_counts.append(avg_generations)
    intermediate_best_distances.append(avg_best_intermediate_best_dist)

def print_metric_stats(name, values):
    print(f"{name}: mean = {np.mean(values):.2f}, std = {np.std(values):.2f}")
# print_metric_stats("Time", full_times)
# print_metric_stats("Coverage", coverages)
# print_metric_stats("Proximity", distances)
# print_metric_stats("Sparsity", sparsities)
# print_metric_stats("Inrermediate best distance", intermediate_best_distances)

## Random population

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    "age":'i',
    "statussex":'i',
    "foreignworker":'i'
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints

results_incremental_explainer_random = []
for i in range(5):
    import time
    strategy = "new_random_population"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="new_random_population", population_size_dynamic=0, cfes_requested=1,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_random.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/GermanCredit/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_random, open(f'{results_dir}/results_incremental{strategy}.pkl', 'wb'))

100%|██████████| 166/166 [01:16<00:00,  2.18it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [01:14<00:00,  2.22it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [01:13<00:00,  2.25it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [01:12<00:00,  2.29it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [01:15<00:00,  2.19it/s]

Empty intermediate counter: 0


In [ ]:
total_time_dynamic_arr_for_ttest_random=[]
total_generations_arr_for_ttest_random=[]
total_cfes_found_arr_for_ttest_random=[]
total_proximity_loss_arr_for_ttest_random=[]
total_sparsity_arr_for_ttest_random=[]
total_best_intermediate_best_dist_arr_for_ttest_random=[]

full_times_random = []
coverages_random = []
distances_random = []
l1s_random = []
proximities_random = []
sparsities_random = []
generation_counts_random = []
intermediate_best_distances_random = []

for explainer in results_incremental_explainer_random:
    forttest, foravgs = stats_incremental(iea, explainer, return_matrices=True)
    time_dynamic_arr, generations_arr, cfes_found_arr, proximity_loss_arr, sparsity_arr, best_intermediate_best_dist_arr = forttest
    time_dynamic, avg_generations, avg_cfes_found, avg_l2, avg_l1, avg_proximity_loss, avg_sparsity, avg_best_intermediate_best_dist = foravgs
    
    total_time_dynamic_arr_for_ttest_random.extend(time_dynamic_arr)
    total_generations_arr_for_ttest_random.extend(generations_arr)
    total_cfes_found_arr_for_ttest_random.extend(cfes_found_arr)
    total_proximity_loss_arr_for_ttest_random.extend(proximity_loss_arr)
    total_sparsity_arr_for_ttest_random.extend(sparsity_arr)
    total_best_intermediate_best_dist_arr_for_ttest_random.extend(best_intermediate_best_dist_arr)

    full_times_random.append(time_dynamic)
    coverages_random.append(avg_cfes_found)
    distances_random.append(avg_l2)
    l1s_random.append(avg_l1)
    proximities_random.append(avg_proximity_loss)
    sparsities_random.append(avg_sparsity)
    generation_counts_random.append(avg_generations)
    intermediate_best_distances_random.append(avg_best_intermediate_best_dist)

def print_metric_stats(name, values):
    print(f"{name}: mean = {np.mean(values):.4f}, std = {np.std(values):.4f}")

In [ ]:
from scipy.stats import ttest_ind
import numpy as np

from scipy.stats import ttest_ind
import numpy as np

def format_stat(value):
    """Format a statistic to 2 decimal places"""
    return f"{value:.2f}"

def format_pvalue(pvalue):
    """Format p-value with scientific notation"""
    if pvalue < 1e-100:
        return "0.0\\times 10^{0}"
    
    sci_notation = f"{pvalue:.1e}".split('e')
    base = float(sci_notation[0])
    exponent = int(sci_notation[1])
    
    return f"{base:.1f}\\times 10^{{{exponent}}}"


# Perform t-tests 
ttest_times = ttest_ind(total_time_dynamic_arr_for_ttest, total_time_dynamic_arr_for_ttest_random)
ttest_generations = ttest_ind(total_generations_arr_for_ttest, total_generations_arr_for_ttest_random)
ttest_cfes_found = ttest_ind(total_cfes_found_arr_for_ttest, total_cfes_found_arr_for_ttest_random)
ttest_distances = ttest_ind(total_proximity_loss_arr_for_ttest, total_proximity_loss_arr_for_ttest_random)
ttest_sparsity = ttest_ind(total_sparsity_arr_for_ttest, total_sparsity_arr_for_ttest_random)
test_distances_intermediate = ttest_ind(total_best_intermediate_best_dist_arr_for_ttest, total_best_intermediate_best_dist_arr_for_ttest_random)

warm_means = [
    np.mean(full_times),
    np.mean(generation_counts), 
    np.mean(coverages),
    np.mean(proximities),
    np.mean(sparsities),
    # np.mean(intermediate_best_distances)
]

random_means = [
    np.mean(full_times_random),
    np.mean(generation_counts_random),
    np.mean(coverages_random),
    np.mean(proximities_random),
    np.mean(sparsities_random),
    # np.mean(intermediate_best_distances_random),
]

pvalues = [
    ttest_times.pvalue,
    ttest_generations.pvalue, 
    ttest_cfes_found.pvalue,
    ttest_distances.pvalue,
    ttest_sparsity.pvalue,
    # test_distances_intermediate.pvalue
]

rows = [
    f"\\shortstack{{\\texttt{{Compas}}}} & {' & '.join(f'${format_pvalue(p)}$' for p in pvalues)} \\\\",
    f"Warm & {' & '.join(f'${format_stat(m)}$' for m in warm_means)} \\\\",
    f"Random & {' & '.join(f'${format_stat(m)}$' for m in random_means)} \\\\"
]

print("\n".join(rows))

\shortstack{\texttt{Compas}} & $0.0\times 10^{0}$ & $3.5\times 10^{-20}$ & $3.9\times 10^{-1}$ & $1.8\times 10^{-5}$ & $3.4\times 10^{-3}$ \\
Warm & $18.27$ & $6.03$ & $95.36$ & $0.11$ & $0.02$ \\
Random & $33.89$ & $6.36$ & $96.07$ & $0.13$ & $0.02$ \\
